In [ ]:
from gym_interface import Agent, State
import copy
import math
import random
import socket
import time
from collections import deque, namedtuple
from typing import Dict, Iterable, List, Literal, Optional, Union

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from datetime import datetime
import shutil as shut
from distutils.util import strtobool
from rlmodel.utils.utils import print_args, print_box, connected_to_internet
import wandb
import setproctitle
from pathlib import Path

import os, sys
from rlmodel.onpolicy.runner.shared.mpe_runner import MPERunner as Runner
from concurrent.futures import ThreadPoolExecutor

*自定义处理函数*

In [ ]:
def parse_Input(action) -> str:
    # example:
    fCmdHeadingDeg1 = [-60.0, -30.0, 0.0, 30.0, 60.0, 90.0, -90.0]
    fCmdHeadingDeg2 = [-60.0, -30.0, 0.0, 30.0, 60.0, 90.0, -90.0]
    fCmdHeadingDeg3 = [-60.0, -30.0, 0.0, 30.0, 60.0, 90.0, -90.0]
    iTurnDirection1 = [-1, -1, 0, 1, 1, 1, -1]
    iTurnDirection2 = [-1, -1, 0, 1, 1, 1, -1]
    iTurnDirection3 = [-1, -1, 0, 1, 1, 1, -1,]
    hover = {
                "iCmdID": 5,
                "bApply": 0,
                "fCmdAlt": 0,
                "fCmdThrust": 7,
                "fCmdHeadingDeg": 0,
                "fCmdPitchDeg": 0,
                "fCmdDotphi": 0,
                "fCmdMa": 0,
                "fCmdPhiDeg": 0,
                "fCmdObliqueDeg": 0,
                "fCmdNy": 7,
                "iTurnDirection": 1,
                "iCmdIndex": 0,
            }
    s = {}
    if(action[0,0]==7):
        s["Beh_outinfo1"] = hover
    else:
        s["Beh_outinfo1"] = {
                "iCmdID": 10,
                "bApply": 0,
                "fCmdAlt": 0,
                "fCmdThrust": 5,
                "fCmdHeadingDeg": fCmdHeadingDeg1[action[0, 0]],
                "fCmdPitchDeg": 0,
                "fCmdDotphi": 0,
                "fCmdMa": 0,
                "fCmdPhiDeg": 0,
                "fCmdObliqueDeg": 0,
                "fCmdNy": 6,
                "iTurnDirection": iTurnDirection1[action[0, 0]],
                "iCmdIndex": 0,
    }
    
    if(action[1,0]==7):
        s["Beh_outinfo2"] = hover
    else:
        s["Beh_outinfo2"] = {
                "iCmdID": 10,
                "bApply": 0,
                "fCmdAlt": 0,
                "fCmdThrust": 5,
                "fCmdHeadingDeg": fCmdHeadingDeg2[action[1, 0]],
                "fCmdPitchDeg": 0,
                "fCmdDotphi": 0,
                "fCmdMa": 0,
                "fCmdPhiDeg": 0,
                "fCmdObliqueDeg": 0,
                "fCmdNy": 6,
                "iTurnDirection": iTurnDirection2[action[1, 0]],
                "iCmdIndex": 0,
            }
    
    if(action[2,0]==7):
        s["Beh_outinfo3"] = hover
    else:
        s["Beh_outinfo3"] = {
                "iCmdID": 10,
                "bApply": 0,
                "fCmdAlt": 0,
                "fCmdThrust": 5,
                "fCmdHeadingDeg": fCmdHeadingDeg3[action[2, 0]],
                "fCmdPitchDeg": 0,
                "fCmdDotphi": 0,
                "fCmdMa": 0,
                "fCmdPhiDeg": 0,
                "fCmdObliqueDeg": 0,
                "fCmdNy": 6,
                "iTurnDirection": iTurnDirection3[action[2, 0]],
                "iCmdIndex": 0,
            }
        
    return s

In [ ]:
def parse_Output(state: Dict[str, any]) -> dict:
    # example:
    tmp1 = []
    tmp2 = []
    tmp3 = []
    Input = {}
    for input in state:
        for k, v in input.items():
            if k == "Longitude":
                tmp1.append(v)
            elif k == "Latitude":
                tmp2.append(v)
            elif k == "Altitude":
                tmp3.append(v)
            else:
                Input[k] = v
    Input["Longitude"] = tmp1
    Input["Latitude"] = tmp2
    Input["Altitude"] = tmp3
    return Input

In [ ]:
def outputToTensor(state: Dict[str, any], i: int = 0) -> np.array:
    self_lon = state["Longitude"][i]
    self_lat = state["Latitude"][i]
    return np.array([self_lon, self_lat])

In [ ]:
def cal_Reward(state: Dict[str, any], i: int = 0):
    self_lon = state["Longitude"][i]
    self_lat = state["Latitude"][i]
    enemy_lon = [93.0, 93.0, 93.0]
    enemy_lat = [0.0, 0.5, 1.0]

    dist_to_goal1 = np.sqrt(
        np.square(self_lon - enemy_lon[0])
        + np.square(self_lat - enemy_lat[0])
    )
    dist_to_goal2 = np.sqrt(
        np.square(self_lon - enemy_lon[1])
        + np.square(self_lat - enemy_lat[1])
    )
    dist_to_goal3 = np.sqrt(
        np.square(self_lon - enemy_lon[2])
        + np.square(self_lat - enemy_lat[2])
    )
    dist_to_goal = min(dist_to_goal1,dist_to_goal2,dist_to_goal3)
    r = -dist_to_goal
    
    if abs(dist_to_goal) < 0.05:
        r += 10
    elif abs(dist_to_goal) < 0.1:
        r += 5
    return r

In [ ]:
def cal_Termination(state: Dict[str, any]) -> bool:
    enemy_lon = [93.0, 93.0, 93.0]
    enemy_lat = [0.0, 0.5, 1.0]
    cnt = 0

    dist_to_goal11 = np.sqrt(
        np.square(state["Longitude"][0] - enemy_lon[0])
        + np.square(state["Latitude"][0] - enemy_lat[0])
    )
    dist_to_goal12 = np.sqrt(
        np.square(state["Longitude"][0] - enemy_lon[1])
        + np.square(state["Latitude"][0] - enemy_lat[1])
    )
    dist_to_goal13 = np.sqrt(
        np.square(state["Longitude"][0] - enemy_lon[2])
        + np.square(state["Latitude"][0] - enemy_lat[2])
    )

    dist_to_goal21 = np.sqrt(
        np.square(state["Longitude"][1] - enemy_lon[0])
        + np.square(state["Latitude"][1] - enemy_lat[0])
    )
    dist_to_goal22 = np.sqrt(
        np.square(state["Longitude"][1] - enemy_lon[1])
        + np.square(state["Latitude"][1] - enemy_lat[1])
    )
    dist_to_goal23 = np.sqrt(
        np.square(state["Longitude"][1] - enemy_lon[2])
        + np.square(state["Latitude"][1] - enemy_lat[2])
    )

    dist_to_goal31 = np.sqrt(
        np.square(state["Longitude"][2] - enemy_lon[0])
        + np.square(state["Latitude"][2] - enemy_lat[0])
    )
    dist_to_goal32 = np.sqrt(
        np.square(state["Longitude"][2] - enemy_lon[1])
        + np.square(state["Latitude"][2] - enemy_lat[1])
    )
    dist_to_goal33 = np.sqrt(
        np.square(state["Longitude"][2] - enemy_lon[2])
        + np.square(state["Latitude"][2] - enemy_lat[2])
    )

    dist_to_goal1 = min(dist_to_goal11,dist_to_goal12,dist_to_goal13)
    dist_to_goal2 = min(dist_to_goal21,dist_to_goal22,dist_to_goal23)
    dist_to_goal3 = min(dist_to_goal31,dist_to_goal32,dist_to_goal33)

    if dist_to_goal1 < 0.05:
        cnt += 1
    if dist_to_goal2 < 0.05:
        cnt += 1
    if dist_to_goal3 < 0.05:
        cnt += 1

    # if cnt > 0:
    #     shut.copytree(
    #         "D:\\postgraduate\\FinalProject\\simplecq-master\\data",
    #         "D:\\postgraduate\\FinalProject\\simplecq-master\\images/"
    #         + datetime.now().strftime("%Y%m%d-%H%M%S")
    #         + "/data",
    #     )

    if cnt>=3:
        print("===========\n")
        print("===========\n")
        print("===========\n")
        shut.copytree(
            "D:\\postgraduate\\FinalProject\\simplecq-master\\data",
            "D:\\postgraduate\\FinalProject\\simplecq-master\\images/"
            + datetime.now().strftime("%Y%m%d-%H%M%S")
            + "/data",
        )
        return True

    else:

        return False
    
def cal_obs(state: Dict[str, any]):
    enemy_lon = [93.0, 93.0, 93.0]
    enemy_lat = [0.0, 0.5, 1.0]
    
    dist_to_goal11 = np.sqrt(
        np.square(state["Longitude"][0] - enemy_lon[0])
        + np.square(state["Latitude"][0] - enemy_lat[0])
    )
    dist_to_goal12 = np.sqrt(
        np.square(state["Longitude"][0] - enemy_lon[1])
        + np.square(state["Latitude"][0] - enemy_lat[1])
    )
    dist_to_goal13 = np.sqrt(
        np.square(state["Longitude"][0] - enemy_lon[2])
        + np.square(state["Latitude"][0] - enemy_lat[2])
    )
    
    dist_to_goal21 = np.sqrt(
        np.square(state["Longitude"][1] - enemy_lon[0])
        + np.square(state["Latitude"][1] - enemy_lat[0])
    )
    dist_to_goal22 = np.sqrt(
        np.square(state["Longitude"][1] - enemy_lon[1])
        + np.square(state["Latitude"][1] - enemy_lat[1])
    )
    dist_to_goal23 = np.sqrt(
       np.square(state["Longitude"][1] - enemy_lon[2])
        + np.square(state["Latitude"][1] - enemy_lat[2])
    )

    dist_to_goal31 = np.sqrt(
        np.square(state["Longitude"][2] - enemy_lon[0])
        + np.square(state["Latitude"][2] - enemy_lat[0])
    )
    dist_to_goal32 = np.sqrt(
        np.square(state["Longitude"][2] - enemy_lon[1])
        + np.square(state["Latitude"][2] - enemy_lat[1])
    )
    dist_to_goal33 = np.sqrt(
        np.square(state["Longitude"][2] - enemy_lon[2])
        + np.square(state["Latitude"][2] - enemy_lat[2])
    )

    dist_to_goal1 = min(dist_to_goal11,dist_to_goal12,dist_to_goal13)
    dist_to_goal2 = min(dist_to_goal21,dist_to_goal22,dist_to_goal23)
    dist_to_goal3 = min(dist_to_goal31,dist_to_goal32,dist_to_goal33)
    
    return dist_to_goal1 < 0.05 , dist_to_goal2 < 0.05 , dist_to_goal3 < 0.05


_算法参数配置_

In [ ]:
device = torch.device("cuda")
sys.path.append(os.path.abspath(os.getcwd()))
num_agents = 3
num_enemies = 1
episode_length = 50
decision_time = 200
save_interval = 1000
log_interval = 10
model_dir = Path(os.path.dirname(os.path.dirname(os.getcwd())) + "/results") / "save"
all_args = {
    "algorithm_name": "rmappo",
    "use_recurrent_policy": True,
    "use_naive_recurrent_policy": False,
    "share_policy": True,
    "use_wandb": True,
    "seed": 0,
    "use_centralized_V": True,
    "use_linear_lr_decay": True,
    "hidden_size": 32,
    "recurrent_N": 1,
    "act_space": 5,
    "obs_space": 2,
    "shared_obs_space": 2 * num_agents,
    "model_dir": None,
    "episode_length": episode_length,
    "gamma": 0.98,
    "gae_lambda": 0.95,
    "use_gae": True,
    "clip_param": 0.2,
    "ppo_epoch": 15,
    "num_mini_batch": 1,
    "data_chunk_length": 10,
    "value_loss_coef": 0.5,
    "entropy_coef": 0.01,
    "max_grad_norm": 10.0,
    "huber_delta": 10.0,
    "use_max_grad_norm": True,
    "use_clipped_value_loss": True,
    "use_huber_loss": True,
    "use_popart": True,
    "use_valuenorm": False,
    "use_value_active_masks": True,
    "use_policy_active_masks": True,
    "lr": 7e-7,
    "critic_lr": 7e-5,
    "opti_eps": 1e-5,
    "weight_decay": 0,
    "gain": 0.01,
    "use_orthogonal": True,
    "use_feature_normalization": True,
    "use_ReLU": False,
    "stacked_frames": 1,
    "layer_N": 1,
    "n_rollout_threads": 5,
}


run_dir = (
    Path(os.path.dirname(os.path.dirname(os.getcwd())) + "/results")
    / all_args["algorithm_name"]
)
config = {
    "all_args": all_args,
    "num_agents": num_agents,
    "num_enemies": num_enemies,
    "device": device,
    "run_dir": run_dir,
}

*W&B记录训练日志*

In [ ]:
if all_args["use_wandb"]:
        # for supercloud when no internet_connection
        if not connected_to_internet():
            import json

            # save a json file with your wandb api key in your
            # home folder as {'my_wandb_api_key': 'INSERT API HERE'}
            # NOTE this is only for running on systems without internet access
            # have to run `wandb sync wandb/run_name` to sync logs to wandboard
            with open(os.path.split(os.getcwd())[0] + "/keys.json") as json_file:
                key = json.load(json_file)
                my_wandb_api_key = key["my_wandb_api_key"]  # NOTE change here as well
            os.environ["WANDB_API_KEY"] = my_wandb_api_key
            os.environ["WANDB_MODE"] = "dryrun"
            os.environ["WANDB_SAVE_CODE"] = "true"

        print_box("Creating wandboard...")
        run = wandb.init(
            config=all_args,
            project="simplecq",
            # project=all_args.env_name,
            # entity="cc",
            notes=socket.gethostname(),
            name=str(all_args["algorithm_name"])
            + "_seed"
            + str(all_args["seed"]) + "MP",
            # group=all_args.scenario_name,
            dir=str(run_dir),
            # job_type="training",
            reinit=True,
        )
        
setproctitle.setproctitle(
        str(all_args["algorithm_name"])
        + "@"
        + str("lapluma030")
    )

# seed
torch.manual_seed(all_args["seed"])
torch.cuda.manual_seed_all(all_args["seed"])
np.random.seed(all_args["seed"])

In [ ]:
runner = Runner(config)
print_box("Actor Network", 80)
if type(runner.policy) == list:
    print_box(runner.policy[0].actor, 80)
    print_box("Critic Network", 80)
    print_box(runner.policy[0].critic, 80)
else:
    print_box(runner.policy.actor, 80)
    print_box("Critic Network", 80)
    print_box(runner.policy.critic, 80)



*端口及输出类型指定*

In [ ]:
port = 40029
outputs_type = {
    "Beh_outinfo1": {
        "iCmdID": "uint64_t",
        "bApply": "bool",
        "fCmdAlt": "double",
        "fCmdThrust": "double",
        "fCmdHeadingDeg": "double",
        "fCmdPitchDeg": "double",
        "fCmdDotphi": "double",
        "fCmdMa": "double",
        "fCmdPhiDeg": "double",
        "fCmdObliqueDeg": "double",
        "fCmdNy": "double",
        "iTurnDirection": "int32_t",
        "iCmdIndex": "uint64_t",
    },
    "Beh_outinfo2": {
        "iCmdID": "uint64_t",
        "bApply": "bool",
        "fCmdAlt": "double",
        "fCmdThrust": "double",
        "fCmdHeadingDeg": "double",
        "fCmdPitchDeg": "double",
        "fCmdDotphi": "double",
        "fCmdMa": "double",
        "fCmdPhiDeg": "double",
        "fCmdObliqueDeg": "double",
        "fCmdNy": "double",
        "iTurnDirection": "int32_t",
        "iCmdIndex": "uint64_t",
    },
    "Beh_outinfo3": {
        "iCmdID": "uint64_t",
        "bApply": "bool",
        "fCmdAlt": "double",
        "fCmdThrust": "double",
        "fCmdHeadingDeg": "double",
        "fCmdPitchDeg": "double",
        "fCmdDotphi": "double",
        "fCmdMa": "double",
        "fCmdPhiDeg": "double",
        "fCmdObliqueDeg": "double",
        "fCmdNy": "double",
        "iTurnDirection": "int32_t",
        "iCmdIndex": "uint64_t",
    },
}

In [ ]:
agents = []
for i in range(all_args["n_rollout_threads"]):
    agents.append(Agent(port=port+i, 
                outputs_type=outputs_type,
                process_input=parse_Input,
                process_output=parse_Output,
                reward_func=cal_Reward,
                end_func=cal_Termination))


*训练函数*

In [ ]:
def train(env: List[Agent], episodes, enable_log=True):
    tag11 = tag12 = tag13 = False
    tag21 = tag22 = tag23 = False
    tag31 = tag32 = tag33 = False
    tag41 = tag42 = tag43 = False
    tag51 = tag52 = tag53 = False

    n_envs = all_args["n_rollout_threads"]
    # 初始化环境
    with ThreadPoolExecutor() as executor:
        tasks = [executor.submit(env[i].reset) for i in range(n_envs)]
        results = [task.result() for task in tasks]

    # 初始化 obs、reward、done
    s = np.zeros((n_envs, num_agents, all_args["obs_space"]))
    r = np.zeros((n_envs, num_agents, 1))
    dones = np.full((n_envs, num_agents), False)

    # replay buffer 初始化
    if runner.use_centralized_V:
        share_obs = s.reshape(n_envs, -1)
        share_obs = np.expand_dims(share_obs, 1).repeat(runner.num_agents, axis=1)
    else:
        share_obs = s

    runner.buffer.share_obs[0] = share_obs.copy()
    runner.buffer.obs[0] = s.copy()

    # 每个环境单独计步
    env_step_count = np.zeros(n_envs, dtype=np.int32)
    max_env_steps = 15000  # 每个环境最大对局步数

    for episode in range(episodes):
        print(f"episode:{episode}/{episodes}")

        if all_args["use_linear_lr_decay"]:
            runner.trainer.policy.lr_decay(episode, episodes)

        for step in range(episode_length * decision_time):
            # Sample actions
            if (step % decision_time) == 0:
                (
                    values,
                    actions,
                    action_log_probs,
                    rnn_states,
                    rnn_states_critic,
                    actions_env,
                ) = runner.collect(step // decision_time)

                act = copy.deepcopy(actions)
                # 自定义 tag 动作修改
                if tag11: act[0,0,0] = 7
                if tag12: act[0,1,0] = 7
                if tag13: act[0,2,0] = 7
                if tag21: act[1,0,0] = 7
                if tag22: act[1,1,0] = 7
                if tag23: act[1,2,0] = 7
                if tag31: act[2,0,0] = 7
                if tag32: act[2,1,0] = 7
                if tag33: act[2,2,0] = 7
                if tag41: act[3,0,0] = 7
                if tag42: act[3,1,0] = 7
                if tag43: act[3,2,0] = 7
                if tag51: act[4,0,0] = 7
                if tag52: act[4,1,0] = 7
                if tag53: act[4,2,0] = 7
                # 同步 step 多环境
                with ThreadPoolExecutor() as executor:
                    tasks = [executor.submit(env[i].step, act[i]) for i in range(n_envs)]
                    results = [task.result() for task in tasks]

                obs, rew, done, _ = zip(*results)

                # 更新 tag
                tag11, tag12, tag13 = cal_obs(obs[0])
                tag21, tag22, tag23 = cal_obs(obs[1])
                tag31, tag32, tag33 = cal_obs(obs[2])
                tag41, tag42, tag43 = cal_obs(obs[3])
                tag51, tag52, tag53 = cal_obs(obs[4])
                # 转换 obs/reward
                s = np.array([[outputToTensor(obs[i], j) for j in range(num_agents)] for i in range(n_envs)])
                r = np.array([[[cal_Reward(obs[i], j)] for j in range(num_agents)] for i in range(n_envs)])
                dones = np.array([[done[i] for _ in range(num_agents)] for i in range(n_envs)])

                # insert data into buffer
                data = (s, r, dones, values, actions, action_log_probs, rnn_states, rnn_states_critic)
                runner.insert(data)

                # env 步数累加
                env_step_count += 1

                # 异步 per-env reset
                for i in range(n_envs):
                    if done[i] or env_step_count[i] >= max_env_steps:
                        env_step_count[i] = 0
                        env[i].reset()
                        obs_i = [(92,0), (92,0.5), (92,1)]
                        # 更新 obs / s / share_obs
                        s[i] = np.array(obs_i)
                        if runner.use_centralized_V:
                            share_obs[i] = s[i].reshape(1, -1).repeat(num_agents, axis=0)
                        else:
                            share_obs[i] = s[i]

            else:
                # else 分支：物理模型缓存动作
                if tag11: act[0,0,0] = 7
                if tag12: act[0,1,0] = 7
                if tag13: act[0,2,0] = 7
                if tag21: act[1,0,0] = 7
                if tag22: act[1,1,0] = 7
                if tag23: act[1,2,0] = 7
                if tag31: act[2,0,0] = 7
                if tag32: act[2,1,0] = 7
                if tag33: act[2,2,0] = 7
                if tag41: act[3,0,0] = 7
                if tag42: act[3,1,0] = 7
                if tag43: act[3,2,0] = 7
                if tag51: act[4,0,0] = 7
                if tag52: act[4,1,0] = 7
                if tag53: act[4,2,0] = 7
                with ThreadPoolExecutor() as executor:
                    tasks = [executor.submit(env[i].step, act[i]) for i in range(n_envs)]
                    results = [task.result() for task in tasks]

                obs, rew, done, _ = zip(*results)
                tag11, tag12, tag13 = cal_obs(obs[0])
                tag21, tag22, tag23 = cal_obs(obs[1])
                tag31, tag32, tag33 = cal_obs(obs[2])
                tag41, tag42, tag43 = cal_obs(obs[3])
                tag51, tag52, tag53 = cal_obs(obs[4])
                # env 步数累加
                env_step_count += 1
                # 异步 per-env reset
                for i in range(n_envs):
                    if done[i] or env_step_count[i] >= max_env_steps:
                        env_step_count[i] = 0
                        env[i].reset()
                        obs_i = [(92,0), (92,0.5), (92,1)]
                        s[i] = np.array(obs_i)
                        if runner.use_centralized_V:
                            share_obs[i] = s[i].reshape(1, -1).repeat(num_agents, axis=0)
                        else:
                            share_obs[i] = s[i]

            # 训练
            if (step % decision_time) == 0:
                runner.compute()
                train_infos = runner.train()

        # post process
        total_num_steps = (episode + 1) * episode_length * n_envs

        # save model
        if episode % save_interval == 0 or episode == episodes - 1:
            runner.save()

        # log information
        if episode % log_interval == 0:
            avg_ep_rew = np.mean(runner.buffer.rewards) * episode_length
            train_infos["average_episode_rewards"] = avg_ep_rew
            print(
                f"Average episode rewards is {avg_ep_rew:.3f} \t"
                f"Total timesteps: {total_num_steps} \t "
                f"Percentage complete {total_num_steps / episodes / episode_length / n_envs * 100:.3f}"
            )
            runner.log_train(train_infos, total_num_steps)

        if episode % 1000 == 0 and enable_log:
            print(f"finished: {episode / episodes * 100}%")

    if all_args["use_wandb"]:
        run.finish()


*运行训练*

In [ ]:
train(agents, 6000)